# CareSignal — GPU fine-tuning + vLLM benchmark (self-contained)

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** (~45 min).
When it finishes, the last cell downloads `gpu_results.zip`; send that file back.

1. Fine-tune RoBERTa-base on PubMed 20k RCT sentences on the GPU (fp16): 8k sentences (same setup as the CPU run) and 50k sentences.
2. Benchmark LLM serving: vLLM vs Hugging Face `generate`, same model and prompts.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
!pip install -q mlflow
!mkdir -p data/raw reports
!for s in train dev test; do curl -fsSL -o data/raw/pubmed20k_$s.txt https://raw.githubusercontent.com/Franck-Dernoncourt/pubmed-rct/master/PubMed_20k_RCT/$s.txt; done
!wc -l data/raw/*

In [ ]:
%%writefile finetune_rct.py
"""Fine-tune RoBERTa-base to label sentences of randomized-controlled-trial abstracts
(BACKGROUND / OBJECTIVE / METHODS / RESULTS / CONCLUSIONS) on PubMed 20k RCT.

Runs on CPU or GPU: on CUDA it switches on fp16 autocast and a larger batch.
Everything (params, per-epoch dev metrics, test report, confusion matrix) goes to MLflow,
and the fine-tuned model is logged to the MLflow Model Registry.

    python -m caresignal.nlp.finetune --model models/roberta-base --train-n 8000 --epochs 2
"""
from __future__ import annotations

import argparse
import json
import os
import random
import time
from pathlib import Path

import mlflow
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup



LABELS = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]
URL = "https://raw.githubusercontent.com/Franck-Dernoncourt/pubmed-rct/master/PubMed_20k_RCT/{split}.txt"


def load_split(path: str | Path) -> tuple[list[str], list[int]]:
    texts, labels = [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line or line.startswith("###"):
            continue
        lab, text = line.split("\t", 1)
        texts.append(text)
        labels.append(LABELS.index(lab))
    return texts, labels


def sample(texts: list[str], labels: list[int], n: int | None, seed: int) -> tuple[list[str], list[int]]:
    """Seeded random subset (n=None or n>=len keeps everything)."""
    if n is None or n >= len(texts):
        return texts, labels
    idx = random.Random(seed).sample(range(len(texts)), n)
    return [texts[i] for i in idx], [labels[i] for i in idx]


REG_NAME = os.environ.get("CARESIGNAL_NLP_MODEL_NAME", "caresignal-rct-sentence-roberta")


def batches(texts, labels, tok, bs, max_len, shuffle, seed):
    idx = list(range(len(texts)))
    if shuffle:
        random.Random(seed).shuffle(idx)
    for i in range(0, len(idx), bs):
        j = idx[i:i + bs]
        enc = tok([texts[k] for k in j], padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        yield enc, torch.tensor([labels[k] for k in j])


@torch.no_grad()
def predict(model, tok, texts, device, bs=64, max_len=64, amp=False):
    model.eval()
    out = []
    for enc, _ in batches(texts, [0] * len(texts), tok, bs, max_len, False, 0):
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            out.append(model(**enc).logits.float().argmax(-1).cpu().numpy())
    return np.concatenate(out)


def register(model, tok) -> None:
    """Log the fine-tuned model to the active run and the Model Registry.
    Requirements are pinned explicitly so MLflow does not probe for optional packages."""
    import tokenizers
    import transformers

    reqs = [f"torch=={torch.__version__.split('+')[0]}", f"transformers=={transformers.__version__}",
            f"tokenizers=={tokenizers.__version__}"]
    mlflow.transformers.log_model(transformers_model={"model": model, "tokenizer": tok}, task="text-classification",
                                  name="model", registered_model_name=REG_NAME, pip_requirements=reqs)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--data", default="data/raw")
    ap.add_argument("--train-n", type=int, default=8000)
    ap.add_argument("--dev-n", type=int, default=1000)
    ap.add_argument("--test-n", type=int, default=5000)
    ap.add_argument("--epochs", type=int, default=2)
    ap.add_argument("--lr", type=float, default=3e-5)
    ap.add_argument("--bs", type=int, default=16)
    ap.add_argument("--max-len", type=int, default=64)
    ap.add_argument("--seed", type=int, default=13)
    ap.add_argument("--out", default="models/rct-roberta")
    ap.add_argument("--report", default="reports/nlp_roberta.json")
    a = ap.parse_args()

    random.seed(a.seed)
    np.random.seed(a.seed)
    torch.manual_seed(a.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = device == "cuda"
    if device == "cpu":
        torch.set_num_threads(max(1, torch.get_num_threads()))

    tr_x, tr_y = sample(*load_split(Path(a.data) / "pubmed20k_train.txt"), a.train_n, a.seed)
    dv_x, dv_y = sample(*load_split(Path(a.data) / "pubmed20k_dev.txt"), a.dev_n, a.seed)
    te_x, te_y = sample(*load_split(Path(a.data) / "pubmed20k_test.txt"), a.test_n, a.seed)

    tok = AutoTokenizer.from_pretrained(a.model)
    model = AutoModelForSequenceClassification.from_pretrained(
        a.model, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id={lab: i for i, lab in enumerate(LABELS)}
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=0.01)
    steps = a.epochs * ((len(tr_x) + a.bs - 1) // a.bs)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    mlflow.set_experiment(os.environ.get("CARESIGNAL_EXPERIMENT_DIR", "") + "caresignal-clinical-nlp")
    with mlflow.start_run(run_name="roberta-base-finetune") as run:
        mlflow.log_params({**vars(a), "device": device, "fp16": amp, "base_model": "roberta-base (spaCy en_core_web_trf 3.8.0)",
                           "n_train": len(tr_x), "n_dev": len(dv_x), "n_test": len(te_x)})
        best, step, t0 = -1.0, 0, time.time()
        for ep in range(a.epochs):
            model.train()
            for enc, y in batches(tr_x, tr_y, tok, a.bs, a.max_len, True, a.seed + ep):
                enc = {k: v.to(device) for k, v in enc.items()}
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    loss = model(**enc, labels=y.to(device)).loss
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
                sched.step()
                step += 1
                if step % 50 == 0:
                    mlflow.log_metric("train_loss", float(loss), step=step)
                    print(f"step {step}/{steps} loss {float(loss):.3f} {time.time() - t0:.0f}s", flush=True)
            dev_f1 = f1_score(dv_y, predict(model, tok, dv_x, device, max_len=a.max_len, amp=amp), average="macro")
            mlflow.log_metric("dev_macro_f1", dev_f1, step=ep + 1)
            print(f"epoch {ep + 1} dev macro-F1 {dev_f1:.4f}", flush=True)
            if dev_f1 > best:  # keep the best epoch on dev, never on test
                best = dev_f1
                model.save_pretrained(a.out)
                tok.save_pretrained(a.out)

        model = AutoModelForSequenceClassification.from_pretrained(a.out).to(device)
        pred = predict(model, tok, te_x, device, max_len=a.max_len, amp=amp)
        rep = classification_report(te_y, pred, target_names=LABELS, output_dict=True, digits=4)
        cm = confusion_matrix(te_y, pred).tolist()
        mlflow.log_metrics({"test_macro_f1": rep["macro avg"]["f1-score"], "test_accuracy": rep["accuracy"],
                            "train_minutes": (time.time() - t0) / 60})
        res = {"model": "roberta-base fine-tuned", "best_dev_macro_f1": best, "test": rep, "confusion": cm,
               "labels": LABELS, "minutes": round((time.time() - t0) / 60, 1), "device": device,
               "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None, "fp16": amp,
               "n_train": len(tr_x), "epochs": a.epochs, "batch_size": a.bs}
        Path(a.report).parent.mkdir(parents=True, exist_ok=True)
        Path(a.report).write_text(json.dumps(res, indent=2))
        mlflow.log_artifact(a.report)
        register(model, tok)
        print(json.dumps({"run_id": run.info.run_id, "test_macro_f1": rep["macro avg"]["f1-score"],
                          "test_acc": rep["accuracy"], "best_dev": best}, indent=1))


if __name__ == "__main__":
    main()


## 1. Fine-tune RoBERTa-base on the GPU

In [ ]:
%env MLFLOW_TRACKING_URI=sqlite:///mlflow.db
%env TOKENIZERS_PARALLELISM=false
!python finetune_rct.py --model FacebookAI/roberta-base --train-n 8000 --epochs 2 --bs 32 --out models/rct-roberta-gpu-8k --report reports/nlp_roberta_gpu_8k.json 2>&1 | grep -v -E 'Warning|warn'

In [ ]:
!python finetune_rct.py --model FacebookAI/roberta-base --train-n 50000 --epochs 2 --bs 64 --out models/rct-roberta-gpu-50k --report reports/nlp_roberta_gpu_50k.json 2>&1 | grep -v -E 'Warning|warn'

## 2. LLM serving: vLLM vs Hugging Face generate
If pip says the session must restart, use **Runtime → Restart session**, then run from this cell down again.

In [ ]:
%%writefile vllm_bench.py
"""Benchmark LLM serving: vLLM vs plain Hugging Face `generate`, same model, same prompts, on GPU.

Task: summarise the objective of randomized-trial abstracts from PubMed 20k RCT (test split).
Both engines use greedy decoding and the same max_new_tokens, so the comparison is throughput
(generated tokens per second) and per-request latency, not answer quality.

Results go to reports/vllm_bench.json and to MLflow.

    python scripts/vllm_bench.py --model Qwen/Qwen2.5-1.5B-Instruct --n 200
Needs a CUDA GPU (e.g. a Colab T4) and `pip install vllm`.
"""
from __future__ import annotations

import argparse
import json
import random
import statistics
import time
from pathlib import Path


def abstracts(path: Path, n: int, seed: int = 13) -> list[str]:
    docs, cur = [], []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.startswith("###"):
            if cur:
                docs.append(" ".join(cur))
            cur = []
        elif "\t" in line:
            cur.append(line.split("\t", 1)[1])
    if cur:
        docs.append(" ".join(cur))
    return random.Random(seed).sample(docs, n)


def prompts(tok, docs: list[str]) -> list[str]:
    msgs = [[{"role": "user", "content": "In one sentence, state the objective of this clinical trial.\n\n" + d[:2500]}]
            for d in docs]
    return [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs]


def run_hf(model_id: str, ps: list[str], max_new: int, bs: int) -> dict:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).cuda().eval()
    lat, gen_tokens = [], 0
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for i in range(0, len(ps), bs):
        enc = tok(ps[i:i + bs], return_tensors="pt", padding=True).to("cuda")
        s = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()
        new = out[:, enc.input_ids.shape[1]:]
        gen_tokens += int((new != tok.eos_token_id).sum())
        lat += [time.perf_counter() - s] * len(new)  # batch latency, shared by its requests
    wall = time.perf_counter() - t0
    del model
    torch.cuda.empty_cache()
    return {"engine": "hf-generate", "batch_size": bs, "wall_s": wall, "gen_tokens": gen_tokens,
            "tokens_per_s": gen_tokens / wall, "p50_latency_s": statistics.median(lat)}


def run_vllm(model_id: str, ps: list[str], max_new: int) -> dict:
    from vllm import LLM, SamplingParams

    llm = LLM(model=model_id, dtype="float16", gpu_memory_utilization=0.85, max_model_len=4096)
    sp = SamplingParams(temperature=0.0, max_tokens=max_new)
    t0 = time.perf_counter()
    outs = llm.generate(ps, sp)
    wall = time.perf_counter() - t0
    gen_tokens = sum(len(o.outputs[0].token_ids) for o in outs)
    lat = [o.metrics.finished_time - o.metrics.arrival_time for o in outs if getattr(o, "metrics", None)]
    return {"engine": "vllm", "wall_s": wall, "gen_tokens": gen_tokens, "tokens_per_s": gen_tokens / wall,
            "p50_latency_s": statistics.median(lat) if lat else None,
            "sample_output": outs[0].outputs[0].text.strip()}


def main() -> None:
    import mlflow
    import torch
    from transformers import AutoTokenizer

    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    ap.add_argument("--data", default="data/raw/pubmed20k_test.txt", type=Path)
    ap.add_argument("--n", type=int, default=200)
    ap.add_argument("--max-new", type=int, default=96)
    ap.add_argument("--hf-batch", type=int, default=16)
    a = ap.parse_args()
    assert torch.cuda.is_available(), "needs a CUDA GPU"
    tok = AutoTokenizer.from_pretrained(a.model)
    ps = prompts(tok, abstracts(a.data, a.n))
    hf = run_hf(a.model, ps, a.max_new, a.hf_batch)
    vl = run_vllm(a.model, ps, a.max_new)
    res = {"model": a.model, "gpu": torch.cuda.get_device_name(0), "n_prompts": len(ps), "max_new_tokens": a.max_new,
           "hf": hf, "vllm": vl, "throughput_speedup": vl["tokens_per_s"] / hf["tokens_per_s"]}
    Path("reports").mkdir(exist_ok=True)
    Path("reports/vllm_bench.json").write_text(json.dumps(res, indent=2))
    mlflow.set_experiment("caresignal-llm-serving")
    with mlflow.start_run(run_name="vllm-vs-hf"):
        mlflow.log_params({"model": a.model, "gpu": res["gpu"], "n": len(ps), "max_new": a.max_new})
        mlflow.log_metrics({"hf_tokens_per_s": hf["tokens_per_s"], "vllm_tokens_per_s": vl["tokens_per_s"],
                            "speedup": res["throughput_speedup"]})
        mlflow.log_artifact("reports/vllm_bench.json")
    print(json.dumps(res, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
!pip install -q vllm

In [ ]:
%env MLFLOW_TRACKING_URI=sqlite:///mlflow.db
!python vllm_bench.py --model Qwen/Qwen2.5-1.5B-Instruct --n 200 2>&1 | tail -40

## 3. Results

In [ ]:
import json, glob
for f in sorted(glob.glob('reports/nlp_roberta_gpu_*.json')):
    r = json.load(open(f))
    print(f, '|', r['gpu'], '| train', r['n_train'], '|', r['minutes'], 'min | test macro-F1', round(r['test']['macro avg']['f1-score'], 4), '| acc', round(r['test']['accuracy'], 4))
try:
    b = json.load(open('reports/vllm_bench.json'))
    print(b['gpu'], '| HF', round(b['hf']['tokens_per_s']), 'tok/s | vLLM', round(b['vllm']['tokens_per_s']), 'tok/s | speedup', round(b['throughput_speedup'], 1), 'x')
except FileNotFoundError:
    print('vLLM benchmark did not finish')
!zip -q -r gpu_results.zip reports mlflow.db
from google.colab import files
files.download('gpu_results.zip')